<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-25</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>의미 기반 검색과 VectorDB(Semantic Search & Vector Database)</strong>에 대해 학습합니다.
>텍스트를 임베딩으로 변환하고 Chroma DB에 저장하여 의미 기반 검색을 학습해봅시다.

</br>

<div style="text-align:center">

</div>

In [ ]:
# TODO 0: 환경변수를 로드하고, 이전 실습에서 만든 all_documents를 준비해봅시다.

import os, glob
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader

load_dotenv()

# data/ 폴더의 모든 PDF 로드
all_documents = []
for pdf_path in sorted(glob.glob("data/*.pdf")):
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    for doc in docs:
        doc.metadata["source_file"] = os.path.basename(pdf_path)
    all_documents.extend(docs)

print(f"PDF {len(glob.glob('data/*.pdf'))}개 → {len(all_documents)}페이지 로드 완료")

</br>

# 임베딩 (Embedding)
> 텍스트를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">고차원 벡터</mark>로 변환하여 의미적 유사성을 수치로 계산할 수 있게 합니다.

> 키워드 검색의 가장 큰 한계는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단어가 다르면 검색이 안 된다</mark>는 점입니다.</br></br>
> "매출"로 검색하면 "수익", "revenue", "실적"이 포함된 문서는 찾지 못합니다. 의미 기반 검색은 이 한계를 두 가지 개념으로 극복합니다.</br></br>
> 첫째, **임베딩(Embedding)**은 텍스트를 수백~수천 차원의 숫자 배열(벡터)로 변환합니다. 핵심은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">의미가 비슷한 텍스트는 벡터 공간에서도 가까운 위치에 놓인다</mark>는 것입니다. "매출", "수익", "revenue"는 서로 다른 단어지만, 임베딩 모델은 이들을 비슷한 벡터로 표현합니다.</br></br>
> 둘째, **벡터 유사도(Vector Similarity)**는 두 벡터가 얼마나 가까운지를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">코사인 유사도</mark>로 측정합니다. 코사인 유사도는 벡터의 방향이 얼마나 일치하는지를 -1 ~ 1 범위의 값으로 나타내며, 1에 가까울수록 의미가 유사한 텍스트입니다. 이 두 개념을 결합하면, 단어가 달라도 의미가 같은 문서를 찾을 수 있습니다.</br>

In [ ]:
# TODO 1: 임베딩 모델을 embedding-query로 초기화하고, "빠른 배송 서비스"의 벡터를 생성하여 차원과 앞 5개 값을 출력해봅시다.

</br>

## 임베딩의 핵심 원리

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">특성</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">유사한 의미</td><td>벡터 거리가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">가까움</mark></td></tr>
    <tr><td style="text-align:center">다른 의미</td><td>벡터 거리가 멂</td></tr>
    <tr><td style="text-align:center">동의어 처리</td><td>"매출" ≈ "수익" ≈ "revenue"</td></tr>
    <tr><td style="text-align:center">다국어</td><td>언어가 달라도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">의미가 같으면 유사</mark></td></tr>
  </tbody>
</table>

</br>

# Vector Store (Chroma)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Chroma</mark>는 벡터를 저장하고 유사도 검색을 수행하는 경량 벡터 데이터베이스입니다.

In [ ]:
# TODO 2: 문서 리스트와 임베딩을 전달하여 벡터 데이터베이스를 생성하고, 저장된 문서 수를 출력해봅시다.

</br>

## Retriever (검색기)
> VectorStore에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">유사한 문서를 검색</mark>하는 인터페이스입니다.

In [ ]:
# TODO 3: 벡터 데이터베이스에서 상위 3개를 반환하는 검색기를 생성하고, "배송 혜택은?"으로 검색하여 결과를 출력해봅시다.

</br>

## 키워드 검색 vs 의미 검색 결과 비교

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">쿼리</th>
      <th style="text-align:center">키워드 검색</th>
      <th style="text-align:center">의미 검색</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">"매출"</td><td style="text-align:center">O</td><td style="text-align:center">O</td></tr>
    <tr><td style="text-align:center">"수익"</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">X</mark></td><td style="text-align:center">O</td></tr>
    <tr><td style="text-align:center">"revenue"</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">X</mark></td><td style="text-align:center">O</td></tr>
    <tr><td style="text-align:center">"회사 실적"</td><td style="text-align:center">X</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">O</mark></td></tr>
  </tbody>
</table>

💡search_kwargs
> `k`: 반환할 문서 수, `score_threshold`: 유사도 임계값
> `search_type="mmr"`로 설정하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다양성</mark>을 고려한 검색도 가능합니다.

💡왜 Chroma인가?
> Chroma는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">로컬에서 바로 실행</mark>되는 경량 벡터 DB입니다.
> 프로덕션에서는 Pinecone, Weaviate, Milvus 등의 전용 서비스를 사용합니다.